## Processing Training Data

There is little discussion on how processing training data should work. It is assumed that most teams are using proprietary methods based off of the information in the contest description, but I wanted to share a basic version that I found some success with in the hopes that it could be a launching point for others.

Here, the majority of the code will look very familiar. However, the handling of gaps and large gaps, plus digit-periods, is enhanced beyond some notebooks I've seen. 


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv


In [2]:
filepath = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
df = pd.read_csv(filepath)
df.head()

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


In [3]:
import re

def clean_transliteration(text):

    # 1. Remove specific single punctuation and symbols: !, ?, /, :, — (em dash)
    # Angle brackets < > are handled by the <.*?> pattern later.
    # The period '.' is handled specifically to keep periods between numbers.
    text = re.sub(r'[!?⁄:—]', '', text)

    # Remove periods that are not between two digits.
    # - Periods between numbers (e.g., '0.83333') will be kept.
    # - Periods between letters (e.g., 'KÙ.BABBAR') will be removed.
    # - Periods at the end/start of words or standalone will be removed.
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)
    # 2. Remove patterns: < > (content inside), ˹ ˺ (content inside), [ ] (content inside)
    # The non-greedy quantifier `*?` ensures it matches the smallest possible string.
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'˹.*?˺', '', text)
    text = re.sub(r'\[.*?\]', '', text)

    # 3. Replace subscripted numbers with their non-subscripted counterparts
    subscript_map = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9'
    }
    for sub, normal in subscript_map.items():
        text = text.replace(sub, normal)

    # Clean up any extra spaces that might result from removals (e.g., double spaces to single space)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("Text cleaning function 'clean_transliteration' defined.")

Text cleaning function 'clean_transliteration' defined.


In [4]:
def replace_gaps(text):
    if pd.isna(text): 
        return text
    
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]\s+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\[\.{3}(?:\s+\.{3})+\]', '<big_gap>', text)
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)

    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r' x ', ' <gap> ', text)
    text = re.sub(r'\[…\]', '<big_gap>', text)
    text = re.sub(r'\[\.\.\.\]', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)
    text = re.sub(r'(x)+', '<gap>', text)

    return text

In [5]:
def clean_gaps(text):
  if pd.isna(text): 
        return text
  text = re.sub(r'<gap> x ', ' <gap> ', text)
  text = re.sub(r'x <gap>', ' <gap>', text)
  text = re.sub(r'<gap> <gap>', '<gap>', text)
  text = re.sub(r'(<gap> )+', r'\1', text)
  return text

In [6]:
def clean_translation(text):
    # Normalize all ellipsis types to a single token
    text = text.replace('…', '...')

    # Replace any bracketed gap like [...], [... ...], [....], [.. etc
    text = re.sub(r'\[\s*\.*\s*(?:\.*\s*)*\]', '<gap>', text)

    # Replace any opening bracket followed by dots: [..., [.... etc
    text = re.sub(r'\[\s*\.+', '<gap>', text)

    # Replace standalone ellipses (three or more dots) with <gap>
    text = re.sub(r'\.{3,}', '<gap>', text)

    # Remove any remaining stray closing brackets
    text = text.replace(']', '')

    # Collapse multiple gaps into one
    text = re.sub(r'(?:<gap>\s*){2,}', '<gap> ', text)

    # Clean up spacing
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [7]:
df['cleaned_akkadian'] = df['transliteration'].apply(clean_transliteration)
df['cleaned_akkadian'] = df['cleaned_akkadian'].apply(replace_gaps)
df['cleaned_akkadian'] = df['cleaned_akkadian'].apply(clean_gaps)
df['cleaned_akkadian'] = df['cleaned_akkadian'].apply(clean_gaps) #Twice, because there are some weird double and triple gaps that take replacing.
print("Cleaned Akkadian column added to DataFrame.")


df['cleaned_translation'] = df['translation'].apply(clean_translation)
#The translations are rather clean besides gaps, it's just occasional brackets and such out of place. 

# Display a sample of original and cleaned text
df.head()

Cleaned Akkadian column added to DataFrame.


,oare_id,transliteration,translation,cleaned_akkadian,cleaned_translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,1 TÚG ša qá-tim i-tur4-DINGIR il5-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙBABBAR,<gap> he did not give you a textile. He return...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",KIŠIB šu-(d)ENLÍL DUMU šu-ku-bi-im KIŠIB ṣí-lu...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


In [8]:
df.to_csv('/kaggle/working/trainProcessed.csv', index=False)